# 04. PSMamba self-supervised pretraining, then fine-tuning (v3)

This notebook is the real project. For each dataset it does:
1. **Pretrain** the encoder with PSMamba's idea: an EMA **teacher** sees the whole Global view; **students** see Mid and Local crops cut out of it. **MPA** maps each crop's tokens back onto the Global 14x14 grid so teacher and student can be compared token by token (masked KL loss).
2. **Fine-tune** with labels using *exactly the same recipe as your baseline notebook*, starting from the pretrained teacher encoder.
3. Compare with your from-scratch baseline (PlantDoc 28.97, Citrus 90.61).

**Variants** (set in the settings cell):
- `paper`: as the paper describes (features summed when mapped to the Global grid).
- `improved`: my changes: features **averaged** instead of summed, plus a **global-to-global** loss term.
- `improved_nodelta`: same, without the learned offsets (tests whether they matter).
- `paper_fixedsched`: paper with a fixed Mid/Local weighting (tests the progressive schedule).

**What changed in v3 (and why).** Version 2 collapsed: the teacher outputs went flat (sharpness and prototype use both rising toward 1.0) while the loss fell. v3 follows standard DINO practice: the head is now an MLP followed by L2 normalisation and cosine similarity with the prototypes (bounded logits), the teacher head is an EMA copy, and the teacher temperature is 0.04 from the start. **This deviates from the paper's plain linear head.** It also makes the paper-vs-improved difference smaller, because the L2 normalisation removes much of the effect of summing versus averaging. A collapse guard now stops the run automatically, and k-NN is printed every 2 epochs for the first 10.

**Important, honest notes**
- **This code has not been run yet.** My environment has no torch or GPU, so I only checked the syntax and verified the grid-mapping (MPA) geometry with a separate numpy copy. The quick test below is its first real run. If it errors, send me the full error and I will fix it.
- The paper does not give every detail (head size, schedules, crop boxes). Where it was silent I made a choice, marked in the settings cell.
- Pretraining is slow on a T4 (the paper used 10 GPUs). Crops are reduced to 2 Mid + 4 Local per image. Time the quick test and extrapolate before you start the long runs.
- SSL on about 2k images may not beat training from scratch. If the k-NN accuracy does not rise during pretraining, say so in your report; it is a finding.

Needs: GPU T4 on, Internet on, both datasets attached.

## 1. Install

In [ ]:
!pip install -q mambapy

## 2. Settings

In [ ]:
import os, time, math, random, copy, numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import f1_score
from torchvision.ops import roi_align

QUICK = False      # True = 2 quick epochs just to check everything runs. Set False for the real run.

# ---- fine-tuning: SAME recipe as the baseline notebook, so the comparison is fair ----
EPOCHS = 2 if QUICK else 50
WARMUP = 1 if QUICK else 5
BATCH  = 16
LR, WD = 5e-4, 0.05
DIM, DEPTH = 192, 8

# ---- self-supervised pretraining (PSMamba) ----
SSL_EPOCHS = 2 if QUICK else 50
SSL_WARMUP = 1 if QUICK else 3
BATCH_SSL  = 8                 # same as the paper; use 4 if you get an out-of-memory error
LR_SSL, WD_SSL = 1e-4, 0.05    # paper: AdamW, lr 1e-4
N_MID, N_LOC = 2, 4            # crops per image (paper: 6 and 6, far too slow on one T4)
MID_FRAC = (0.40, 0.70)        # Mid crop side as a fraction of the Global view side
LOC_FRAC = (0.15, 0.35)        # Local crop side as a fraction of the Global view side
K_PROTO = 512                  # number of prototypes (paper does not say; fewer is easier with ~2k images)
TEMP_STUDENT = 0.10            # paper
TEMP_TEACHER = (0.04, 0.04)    # v3: sharp from the start (v2 started soft at 0.07 and collapsed to flat outputs)
CENTER_MOM = 0.90              # paper
EMA_MOM = (0.996, 0.999)       # paper range, cosine ramp
VIS_MIN = 0.25                 # a Global cell counts as "visible" to a crop if its weight mass exceeds this
KNN_EVERY = 10                 # print a k-NN accuracy this often (and every 2 epochs during the first 10)
COLLAPSE_GUARD = True          # stop automatically if outputs go flat for 3 epochs in a row (checked from epoch 6)
PATCH, GLOBAL, GG = 16, 224, 14   # patch size, Global view size, Global token grid (14x14)

VARIANTS = {
    "paper":            dict(norm="sum", use_gg=False, use_delta=True,  progressive=True),
    "improved":         dict(norm="avg", use_gg=True,  use_delta=True,  progressive=True),
    "improved_nodelta": dict(norm="avg", use_gg=True,  use_delta=False, progressive=True),
    "paper_fixedsched": dict(norm="sum", use_gg=False, use_delta=True,  progressive=False),
}

DEV = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEV == "cuda"
torch.backends.cudnn.benchmark = True
SUFFIX = "_quick" if QUICK else ""
RESULTS = f"/kaggle/working/results_ssl{SUFFIX}.csv"
print("device:", DEV, "| ssl epochs:", SSL_EPOCHS, "| finetune epochs:", EPOCHS, "| results:", RESULTS)

## 3. Data (same code and seed as before, so splits are identical)

In [ ]:
import os, random, hashlib, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold

SEED = 42
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")
PD_ROOT = os.environ.get("PD_ROOT", "/kaggle/input/datasets/nirmalsankalana/plantdoc-dataset")
CT_ROOT = os.environ.get("CT_ROOT", "/kaggle/input/datasets/myprojectdictionary/citrus-leaf-disease-image/Citrus Leaf Disease Image")
OUT = os.environ.get("SPLIT_OUT", "/kaggle/working/splits")
os.makedirs(OUT, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)


def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

In [ ]:
def list_images(root):
    rows = []
    for c in sorted(os.listdir(root)):
        p = os.path.join(root, c)
        if not os.path.isdir(p):
            continue
        for f in sorted(os.listdir(p)):
            if f.lower().endswith(IMG_EXT):
                rows.append((os.path.join(p, f), c))
    return pd.DataFrame(rows, columns=["path", "label"])

def safe_split(df, test_size, seed):
    """Stratified split. Classes with fewer than 2 images cannot be stratified, so they stay in the first part."""
    counts = df.label.value_counts()
    rare = counts[counts < 2].index
    df_rare, df_ok = df[df.label.isin(rare)], df[~df.label.isin(rare)]
    a, b = train_test_split(df_ok, test_size=test_size, stratify=df_ok.label, random_state=seed)
    return pd.concat([a, df_rare]).reset_index(drop=True), b.reset_index(drop=True)

_hash_cache = {}
def file_md5(path):
    if path not in _hash_cache:
        with open(path, "rb") as f:
            _hash_cache[path] = hashlib.md5(f.read()).hexdigest()
    return _hash_cache[path]

def drop_leaks(train, val, test):
    """Remove images from train/val that are byte-identical to a test image (and from train if identical to a val image).
    The test set itself is never changed."""
    test_h = {file_md5(p) for p in test.path}
    val = val[~val.path.map(file_md5).isin(test_h)]
    val_h = {file_md5(p) for p in val.path}
    train = train[~train.path.map(file_md5).isin(test_h | val_h)]
    return train.reset_index(drop=True), val.reset_index(drop=True)

# ---------- PlantDoc: use the provided test set; split train 80/20 into train/val ----------
pd_train_full = list_images(f"{PD_ROOT}/train")
pd_test = list_images(f"{PD_ROOT}/test").reset_index(drop=True)
pd_train, pd_val = safe_split(pd_train_full, 0.20, SEED)
n0 = (len(pd_train), len(pd_val))
pd_train, pd_val = drop_leaks(pd_train, pd_val, pd_test)
print(f"PlantDoc: removed {n0[0]-len(pd_train)} train and {n0[1]-len(pd_val)} val images that duplicate a val/test image")
pd_classes = sorted(pd_train_full.label.unique())          # 28 classes (from train)
pd_map = {c: i for i, c in enumerate(pd_classes)}
unknown = set(pd_test.label) - set(pd_map)
assert not unknown, f"test classes missing from train: {unknown}"

pd_train.to_csv(f"{OUT}/plantdoc_train.csv", index=False)
pd_val.to_csv(f"{OUT}/plantdoc_val.csv", index=False)
pd_test.to_csv(f"{OUT}/plantdoc_test.csv", index=False)
print(f"PlantDoc: train {len(pd_train)} | val {len(pd_val)} | test {len(pd_test)} | classes {len(pd_classes)}")

# ---------- Citrus: no provided split, so 5-fold stratified CV (test = 1 fold, val = 12.5% of the rest) ----------
ct = list_images(CT_ROOT)
ct_classes = sorted(ct.label.unique())
ct_map = {c: i for i, c in enumerate(ct_classes)}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
ct_folds = []
for k, (tr_idx, te_idx) in enumerate(skf.split(ct, ct.label)):
    rest, test = ct.iloc[tr_idx], ct.iloc[te_idx].reset_index(drop=True)
    train, val = safe_split(rest, 0.125, SEED)
    train, val = drop_leaks(train, val, test)
    ct_folds.append((train, val, test))
    train.to_csv(f"{OUT}/citrus_fold{k}_train.csv", index=False)
    val.to_csv(f"{OUT}/citrus_fold{k}_val.csv", index=False)
    test.to_csv(f"{OUT}/citrus_fold{k}_test.csv", index=False)
    assert not (set(train.path) & set(test.path)) and not (set(val.path) & set(test.path)) and not (set(train.path) & set(val.path))
print(f"Citrus: {len(ct)} images, classes {ct_classes}")
for k, (tr, va, te) in enumerate(ct_folds):
    print(f"  fold {k}: train {len(tr)} | val {len(va)} | test {len(te)}  | melanose in test: {(te.label == 'Melanose').sum()}")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms as T
from PIL import Image

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = T.Compose([
    T.RandomResizedCrop(224, scale=(0.4, 1.0)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.ColorJitter(0.3, 0.3, 0.3, 0.05),
    T.ToTensor(), T.Normalize(MEAN, STD)])
eval_tf = T.Compose([T.Resize((224, 224)), T.ToTensor(), T.Normalize(MEAN, STD)])

class LeafDS(Dataset):
    def __init__(self, df, cmap, tf):
        self.paths = df.path.tolist()
        self.y = [cmap[l] for l in df.label]
        self.tf = tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.tf(img), self.y[i]

def make_loaders(train_df, val_df, test_df, cmap, bs=32, workers=2, balanced=False):
    tr = LeafDS(train_df, cmap, train_tf)
    sampler = None
    if balanced:   # class-balanced sampling (the paper reports this setting in Table 11)
        cnt = np.bincount(tr.y, minlength=len(cmap)).astype(float)
        w = [1.0 / max(cnt[y], 1) for y in tr.y]
        sampler = WeightedRandomSampler(w, num_samples=len(tr), replacement=True)
    kw = dict(num_workers=workers, pin_memory=torch.cuda.is_available())
    return (DataLoader(tr, bs, shuffle=sampler is None, sampler=sampler, drop_last=True, **kw),
            DataLoader(LeafDS(val_df, cmap, eval_tf), bs, shuffle=False, **kw),
            DataLoader(LeafDS(test_df, cmap, eval_tf), bs, shuffle=False, **kw))

## 4. Encoder (same as the baseline)

In [ ]:
from mambapy.mamba import MambaBlock, MambaConfig

def make_block(d):
    return MambaBlock(MambaConfig(d_model=d, n_layers=1, d_state=16, expand_factor=2,
                                  d_conv=4, pscan=True, use_cuda=False))

class BiMambaBlock(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.norm = nn.LayerNorm(d)
        self.fwd, self.bwd = make_block(d), make_block(d)
        self.wf = nn.Parameter(torch.tensor(0.5))   # learnable fusion weights (paper Eq. 7)
        self.wb = nn.Parameter(torch.tensor(0.5))
    def forward(self, x):
        h = self.norm(x)
        return x + self.wf * self.fwd(h) + self.wb * self.bwd(h.flip(1)).flip(1)

class MiniBiMamba(nn.Module):
    def __init__(self, d=192, depth=8, patch=16, base_grid=14):
        super().__init__()
        self.embed = nn.Conv2d(3, d, patch, patch)
        self.pos = nn.Parameter(torch.zeros(1, d, base_grid, base_grid))
        nn.init.trunc_normal_(self.pos, std=0.02)
        self.blocks = nn.ModuleList([BiMambaBlock(d) for _ in range(depth)])
        self.norm = nn.LayerNorm(d)
    def forward(self, img):
        t = self.embed(img)
        gh, gw = t.shape[-2:]
        pos = F.interpolate(self.pos, size=(gh, gw), mode="bicubic", align_corners=False)
        t = (t + pos).flatten(2).transpose(1, 2)
        for blk in self.blocks:
            t = blk(t)
        return self.norm(t), (gh, gw)

class Classifier(nn.Module):
    def __init__(self, n_classes, d=DIM, depth=DEPTH):
        super().__init__()
        self.enc = MiniBiMamba(d, depth)
        self.head = nn.Linear(d, n_classes)
    def forward(self, x):
        tokens, _ = self.enc(x)
        return self.head(tokens.mean(1))

_m = Classifier(28)
print("Parameters (M):", round(sum(p.numel() for p in _m.parameters()) / 1e6, 2)); del _m

## 5. Fine-tuning code (same recipe as the baseline; can now start from SSL weights)

In [ ]:
def evaluate(model, loader, n_classes):
    model.eval(); ys, ps, loss_sum, n = [], [], 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEV, non_blocking=True), y.to(DEV)
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                out = model(x)
            loss_sum += F.cross_entropy(out.float(), y, reduction="sum").item()
            ps.append(out.argmax(1).cpu()); ys.append(y.cpu()); n += len(y)
    y, p = torch.cat(ys).numpy(), torch.cat(ps).numpy()
    f1s = f1_score(y, p, average=None, labels=list(range(n_classes)), zero_division=0)
    present = np.unique(y)                      # PlantDoc test has one class fewer than train
    return dict(loss=loss_sum / n, acc=float((y == p).mean() * 100),
                macro_f1=float(f1s[present].mean() * 100), f1s=f1s, present=present)

def minority_f1(res, train_df, cmap):
    """Mean F1 of the 25% rarest classes (by training count) that appear in the test set."""
    cnt = train_df.label.map(cmap).value_counts()
    thr = cnt.quantile(0.25)
    minority = [c for c in res["present"] if cnt.get(c, 0) <= thr]
    return float(res["f1s"][minority].mean() * 100) if minority else float("nan")

def get_split(name, fold):
    if name == "plantdoc":
        return pd_train, pd_val, pd_test, pd_map
    tr, va, te = ct_folds[fold]
    return tr, va, te, ct_map

def already_done(tag):
    return os.path.exists(RESULTS) and tag in set(pd.read_csv(RESULTS)["tag"])

def run_experiment(name, fold=0, seed=0, init_path=None, label="ft"):
    tag = f"{label}_{name}_f{fold}_s{seed}"
    if already_done(tag):
        print("skip (already done):", tag); return
    set_seed(seed)
    tr, va, te, cmap = get_split(name, fold)
    ltr, lva, lte = make_loaders(tr, va, te, cmap, bs=BATCH)
    n_cls = len(cmap)
    model = Classifier(n_cls).to(DEV)
    if init_path:                                      # start from the SSL-pretrained (teacher) encoder
        model.enc.load_state_dict(torch.load(init_path, map_location="cpu"))
        print("loaded SSL encoder:", init_path)

    decay = [p for n, p in model.named_parameters() if p.ndim > 1 and not getattr(p, "_no_weight_decay", False) and "pos" not in n]
    no_decay = [p for n, p in model.named_parameters() if not (p.ndim > 1 and not getattr(p, "_no_weight_decay", False) and "pos" not in n)]
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": WD}, {"params": no_decay, "weight_decay": 0.0}], lr=LR)
    total, warm = EPOCHS * len(ltr), WARMUP * len(ltr)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    ck = f"/kaggle/working/ckpt_ft{SUFFIX}_{tag}.pt"
    start, best, hist = 0, dict(val_acc=-1.0, epoch=-1, state=None), []
    if os.path.exists(ck):
        s = torch.load(ck, map_location="cpu", weights_only=False)
        model.load_state_dict(s["model"]); opt.load_state_dict(s["opt"]); sched.load_state_dict(s["sched"])
        scaler.load_state_dict(s["scaler"]); start, best, hist = s["epoch"], s["best"], s["hist"]
        print(f"resumed {tag} from epoch {start}")

    t_start = time.time()
    for ep in range(start, EPOCHS):
        model.train(); t0, tl, nb = time.time(), 0.0, 0
        for x, y in ltr:
            x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                out = model(x)
            loss = F.cross_entropy(out.float(), y, label_smoothing=0.1)
            opt.zero_grad(set_to_none=True)
            if not torch.isfinite(loss):
                sched.step(); continue                 # skip a bad step instead of crashing
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            tl += loss.item(); nb += 1
        r = evaluate(model, lva, n_cls)
        hist.append(dict(epoch=ep, train_loss=tl / max(nb, 1), val_loss=r["loss"], val_acc=r["acc"]))
        if r["acc"] > best["val_acc"]:
            best = dict(val_acc=r["acc"], epoch=ep, state={k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                        scaler=scaler.state_dict(), epoch=ep + 1, best=best, hist=hist), ck + ".tmp")
        os.replace(ck + ".tmp", ck)
        print(f"[{tag}] epoch {ep+1}/{EPOCHS} | train loss {tl/max(nb,1):.3f} | val acc {r['acc']:.2f} | best {best['val_acc']:.2f} (ep {best['epoch']+1}) | {time.time()-t0:.0f}s")

    model.load_state_dict(best["state"])                 # test ONLY with the best-validation weights
    res = evaluate(model, lte, n_cls)
    row = dict(tag=tag, label=label, dataset=name, fold=fold, seed=seed, epochs=EPOCHS, best_epoch=best["epoch"] + 1,
               val_acc=best["val_acc"], test_acc=res["acc"], test_macro_f1=res["macro_f1"],
               test_minority_f1=minority_f1(res, tr, cmap), minutes=(time.time() - t_start) / 60)
    pd.DataFrame([row]).to_csv(RESULTS, mode="a", header=not os.path.exists(RESULTS), index=False)
    print("RESULT", {k: (round(v, 2) if isinstance(v, float) else v) for k, v in row.items()})
    if os.path.exists(ck): os.remove(ck)                # finished: free the space

## 6. PSMamba pretraining code

In [ ]:
class SSLDS(Dataset):
    """Returns two photometric versions of the SAME geometric crop (224x224).
    Teacher sees the weakly augmented one; students see crops cut out of the strongly augmented one.
    Because the geometry is shared, every Mid/Local crop has a known position on the Global 14x14 grid."""
    def __init__(self, df):
        self.paths = df.path.tolist()
        self.geo = T.Compose([T.RandomResizedCrop(224, scale=(0.5, 1.0)), T.RandomHorizontalFlip(), T.RandomVerticalFlip()])
        self.tf_t = T.Compose([T.ColorJitter(0.2, 0.2, 0.2, 0.02), T.ToTensor(), T.Normalize(MEAN, STD)])
        self.tf_s = T.Compose([T.ColorJitter(0.4, 0.4, 0.4, 0.05), T.RandomGrayscale(0.1), T.ToTensor(), T.Normalize(MEAN, STD)])
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = self.geo(Image.open(self.paths[i]).convert("RGB"))
        return self.tf_t(img), self.tf_s(img)

def sample_boxes(B, n, frac, dev):
    """n random square boxes per image, in Global pixel coordinates: (B*n, 3) = x0, y0, side."""
    side = torch.empty(B * n, device=dev).uniform_(*frac) * GLOBAL
    x0 = torch.rand(B * n, device=dev) * (GLOBAL - side)
    y0 = torch.rand(B * n, device=dev) * (GLOBAL - side)
    return torch.stack([x0, y0, side], 1)

def schedules(step, total, progressive):
    p = step / max(1, total)
    tau_t = TEMP_TEACHER[0] + (TEMP_TEACHER[1] - TEMP_TEACHER[0]) * min(1.0, p / 0.3)
    lam_l, lam_m = (1.5 - p, 0.5 + p) if progressive else (1.0, 1.0)   # Local first, Mid later
    mom = EMA_MOM[1] - (EMA_MOM[1] - EMA_MOM[0]) * 0.5 * (1 + math.cos(math.pi * p))
    return dict(tau_t=tau_t, lam_m=lam_m, lam_l=lam_l, lam_g=1.0, mom=mom)

In [ ]:
class DinoHead(nn.Module):
    """MLP -> L2 normalise -> cosine similarity with K prototypes (DINO style).
    Logits are bounded in [-1, 1], so the head cannot shrink its outputs to make both sides flat.
    This replaces the paper's plain linear head, which is what collapsed in my first version."""
    def __init__(self, d, k, hidden=512, bott=128):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(d, hidden), nn.GELU(), nn.Linear(hidden, hidden), nn.GELU(), nn.Linear(hidden, bott))
        self.proto = nn.Parameter(torch.empty(k, bott))
        nn.init.trunc_normal_(self.proto, std=0.02)
    def forward(self, x):
        return F.normalize(self.mlp(x), dim=-1) @ F.normalize(self.proto, dim=-1).T

class PSMambaSSL(nn.Module):
    def __init__(self, d=DIM, depth=DEPTH, k=K_PROTO, use_delta=True):
        super().__init__()
        self.enc = MiniBiMamba(d, depth)                      # student encoder (shared by Global/Mid/Local)
        self.teacher = copy.deepcopy(self.enc)                # EMA teacher
        for p in self.teacher.parameters():
            p.requires_grad = False
        self.head = DinoHead(d, k)                            # student head
        self.teacher_head = copy.deepcopy(self.head)          # EMA teacher head (as in DINO)
        for p in self.teacher_head.parameters():
            p.requires_grad = False
        # scale-specific positional adapters P^(s) (paper Eq. 12), one vector per token position
        self.adapt = nn.ParameterDict({"mid": nn.Parameter(torch.zeros(1, 100, d)),
                                       "loc": nn.Parameter(torch.zeros(1, 36, d))})
        self.use_delta = use_delta
        self.delta = nn.ModuleDict({"mid": nn.Linear(d, 2), "loc": nn.Linear(d, 2)})   # learned offsets (Eq. 9)
        for m in self.delta.values():
            nn.init.zeros_(m.weight); nn.init.zeros_(m.bias)   # start as pure geometry
        self.register_buffer("center", torch.zeros(1, 1, k))

    def transport(self, Z, boxes, grid, key, norm):
        """MPA: move view tokens onto the Global 14x14 grid with bilinear weights.
        Z: (K, grid*grid, d). boxes: (K, 3) = x0, y0, side in Global pixels.
        norm="sum": paper (column-normalised, features are summed per cell).
        norm="avg": my fix (features are averaged per cell), so magnitude no longer depends on zoom."""
        Kc, n, d = Z.shape
        Z = Z.float() + self.adapt[key]
        idx = torch.arange(n, device=Z.device)
        jx, jy = (idx % grid).float(), (idx // grid).float()
        x0, y0, side = boxes[:, 0:1], boxes[:, 1:2], boxes[:, 2:3]
        u = (x0 + (jx[None] + 0.5) * side / grid) / PATCH - 0.5      # x position in Global token units
        v = (y0 + (jy[None] + 0.5) * side / grid) / PATCH - 0.5
        if self.use_delta:
            dlt = 0.25 * torch.tanh(self.delta[key](Z.detach()))
            u, v = u + dlt[..., 0], v + dlt[..., 1]
        u = u.clamp(0, GG - 1 - 1e-4); v = v.clamp(0, GG - 1 - 1e-4)
        u0, v0 = u.floor().detach(), v.floor().detach()
        fu, fv = u - u0, v - v0
        u0, v0 = u0.long(), v0.long()
        out = Z.new_zeros(Kc, GG * GG, d)
        mass = Z.new_zeros(Kc, GG * GG)
        for dy in (0, 1):
            for dx in (0, 1):
                w = (fu if dx else 1 - fu) * (fv if dy else 1 - fv)
                cell = (v0 + dy) * GG + (u0 + dx)
                out = out.scatter_add(1, cell.unsqueeze(-1).expand(-1, -1, d), w.unsqueeze(-1) * Z)
                mass = mass.scatter_add(1, cell, w)
        if norm == "avg":
            out = out / mass.clamp_min(1e-6).unsqueeze(-1)
        return out, mass

    def ssl_loss(self, g_t, g_s, sc, cfg):
        B = g_t.size(0)
        # ---- teacher: sees the whole Global view, no gradient ----
        with torch.no_grad():
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                zt = self.teacher(g_t)[0]
            lt = self.teacher_head(zt.float())                           # (B, 196, K)
            q = F.softmax((lt - self.center) / sc["tau_t"], dim=-1)
            logq = torch.log(q.clamp_min(1e-8))
            ent = float(-(q * logq).sum(-1).mean())                      # per-token sharpness
            pbar = q.mean(dim=(0, 1))                                    # how evenly prototypes are used overall
            marg = float(-(pbar * torch.log(pbar.clamp_min(1e-8))).sum())
            self.center.mul_(CENTER_MOM).add_(lt.mean(dim=(0, 1), keepdim=True), alpha=1 - CENTER_MOM)

        def kl_masked(ls, vis, q_rep, logq_rep):
            logp = F.log_softmax(ls / TEMP_STUDENT, dim=-1)
            kl = (q_rep * (logq_rep - logp)).sum(-1)                     # (K, 196)
            vis = vis.float()
            return ((kl * vis).sum(1) / vis.sum(1).clamp_min(1.0)).mean()  # normalised by |M|_1 (paper Eq. 15)

        losses = {}
        bidx = torch.arange(B, device=g_s.device)
        for key, n, frac, size, grid in (("mid", N_MID, MID_FRAC, 160, 10), ("loc", N_LOC, LOC_FRAC, 96, 6)):
            boxes = sample_boxes(B, n, frac, g_s.device)
            rois = torch.cat([bidx.repeat_interleave(n).float().unsqueeze(1), boxes[:, 0:1], boxes[:, 1:2],
                              boxes[:, 0:1] + boxes[:, 2:3], boxes[:, 1:2] + boxes[:, 2:3]], 1)
            crops = roi_align(g_s.float(), rois, output_size=(size, size), spatial_scale=1.0, sampling_ratio=2, aligned=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                z = self.enc(crops)[0]
            z_on_global, mass = self.transport(z, boxes, grid, key, cfg["norm"])
            ls = self.head(z_on_global)
            losses[key] = kl_masked(ls, mass > VIS_MIN, q.repeat_interleave(n, 0), logq.repeat_interleave(n, 0))
        loss = sc["lam_m"] * losses["mid"] + sc["lam_l"] * losses["loc"]
        if cfg["use_gg"]:                                                # global <-> global term (my addition)
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                zg = self.enc(g_s)[0]
            lg = self.head(zg.float())
            losses["glob"] = kl_masked(lg, torch.ones(B, GG * GG, dtype=torch.bool, device=g_s.device), q, logq)
            loss = loss + sc["lam_g"] * losses["glob"]
        info = {k: float(v.detach()) for k, v in losses.items()}
        info["ent"] = ent
        info["marg"] = marg
        return loss, info

In [ ]:
@torch.no_grad()
def knn_eval(enc, train_df, val_df, cmap, k=20):
    """Quick quality check of the representation: weighted k-NN accuracy on the validation set (no training)."""
    enc.eval()
    def feats(df):
        dl = DataLoader(LeafDS(df, cmap, eval_tf), batch_size=64, shuffle=False, num_workers=2)
        fs, ys = [], []
        for x, y in dl:
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                t = enc(x.to(DEV))[0]
            fs.append(F.normalize(t.float().mean(1), dim=-1).cpu()); ys.append(y)
        return torch.cat(fs), torch.cat(ys)
    ftr, ytr = feats(train_df); fva, yva = feats(val_df)
    top = (fva @ ftr.T).topk(min(k, len(ftr)), dim=1)
    votes = torch.zeros(len(fva), len(cmap))
    votes.scatter_add_(1, ytr[top.indices], (top.values / 0.07).exp())
    return float((votes.argmax(1) == yva).float().mean() * 100)

def pretrain(name, fold=0, seed=0, variant="paper"):
    cfg = VARIANTS[variant]
    tag = f"{variant}_{name}_f{fold}_s{seed}"
    out = f"/kaggle/working/ssl3{SUFFIX}_{tag}.pt"
    if os.path.exists(out):
        print("skip pretraining (already done):", tag); return out
    set_seed(seed)
    tr, va, te, cmap = get_split(name, fold)          # pretraining uses the TRAIN images only (no labels)
    loader = DataLoader(SSLDS(tr), BATCH_SSL, shuffle=True, drop_last=True, num_workers=2, pin_memory=USE_AMP)
    model = PSMambaSSL(use_delta=cfg["use_delta"]).to(DEV)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=LR_SSL, weight_decay=WD_SSL)
    total, warm = SSL_EPOCHS * len(loader), SSL_WARMUP * len(loader)
    lr_sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    ck = f"/kaggle/working/ckpt_ssl3{SUFFIX}_{tag}.pt"
    start, step = 0, 0
    if os.path.exists(ck):
        s = torch.load(ck, map_location="cpu", weights_only=False)
        model.load_state_dict(s["model"]); opt.load_state_dict(s["opt"]); lr_sched.load_state_dict(s["lr"])
        scaler.load_state_dict(s["scaler"]); start, step = s["epoch"], s["step"]
        print(f"resumed {tag} from epoch {start}")
    else:
        print(f"[{tag}] k-NN val acc at random init (reference): {knn_eval(model.teacher, tr, va, cmap):.2f}")

    model.teacher.eval(); model.teacher_head.eval()
    bad = 0
    for ep in range(start, SSL_EPOCHS):
        model.enc.train(); t0 = time.time(); acc, nb = {}, 0
        for g_t, g_s in loader:
            g_t, g_s = g_t.to(DEV, non_blocking=True), g_s.to(DEV, non_blocking=True)
            sc = schedules(step, total, cfg["progressive"])
            loss, info = model.ssl_loss(g_t, g_s, sc, cfg)
            opt.zero_grad(set_to_none=True)
            if not torch.isfinite(loss):
                lr_sched.step(); step += 1; continue
            scaler.scale(loss).backward()
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(params, 3.0)
            scaler.step(opt); scaler.update(); lr_sched.step()
            with torch.no_grad():                                          # EMA teacher update (paper Eq. 17)
                m = sc["mom"]
                t_params = list(model.teacher.parameters()) + list(model.teacher_head.parameters())
                s_params = list(model.enc.parameters()) + list(model.head.parameters())
                for pt, ps in zip(t_params, s_params):
                    pt.mul_(m).add_(ps.detach(), alpha=1 - m)
            for k_, v_ in info.items():
                acc[k_] = acc.get(k_, 0.0) + v_
            nb += 1; step += 1
        msg = " | ".join(f"{k_} {v_/max(nb,1):.3f}" for k_, v_ in acc.items() if k_ not in ("ent", "marg"))
        ent_frac = acc.get("ent", 0.0) / max(nb, 1) / math.log(K_PROTO)
        marg_frac = acc.get("marg", 0.0) / max(nb, 1) / math.log(K_PROTO)
        line = f"[{tag}] epoch {ep+1}/{SSL_EPOCHS} | {msg} | sharpness {ent_frac:.2f} | prototype use {marg_frac:.2f} (0 = collapsed) | {time.time()-t0:.0f}s"
        if (ep + 1) % KNN_EVERY == 0 or ep + 1 == SSL_EPOCHS or ((ep + 1) <= 10 and (ep + 1) % 2 == 0):
            line += f" | kNN val acc {knn_eval(model.teacher, tr, va, cmap):.2f}"
        print(line)
        bad = bad + 1 if (ep + 1 >= 6 and ent_frac > 0.8 and marg_frac > 0.95) else 0
        if COLLAPSE_GUARD and bad >= 3:
            print(f"[{tag}] STOPPED: outputs have been flat for 3 epochs (sharpness {ent_frac:.2f}, prototype use {marg_frac:.2f}). This is collapse, so continuing would waste GPU time.")
            if os.path.exists(ck): os.remove(ck)
            raise RuntimeError("SSL collapsed")
        torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), lr=lr_sched.state_dict(),
                        scaler=scaler.state_dict(), epoch=ep + 1, step=step), ck + ".tmp")
        os.replace(ck + ".tmp", ck)
    torch.save(model.teacher.state_dict(), out)                           # keep the teacher encoder for fine-tuning
    if os.path.exists(ck): os.remove(ck)
    return out

## 7. Quick test (2 epochs each, Citrus fold 0)

Checks that pretraining, the k-NN check and fine-tuning all run. **Ignore the accuracy.** Do look at:
- the time of one SSL epoch (to estimate the real run)
- `prototype use`: this is the collapse check. Near 0 means every token maps to the same prototype (collapsed). A healthy run is clearly above 0.3 and not falling toward 0
- `sharpness`: how peaked each token's teacher distribution is (low is normal at these temperatures)
- that the loss numbers are finite

In [ ]:
ck = pretrain("citrus", fold=0, seed=0, variant="improved")
run_experiment("citrus", fold=0, seed=0, init_path=ck, label="improved")

## 8. Real runs (set `QUICK = False` in cell 2, re-run cells 2 to 6 first)

Start with PlantDoc, where the room is. Run `paper` first (this is the reproduction), then `improved`.

In [ ]:
for v in ("paper",):
    ck = pretrain("plantdoc", fold=0, seed=0, variant=v)
    run_experiment("plantdoc", fold=0, seed=0, init_path=ck, label=v)

Citrus is cheaper but noisy. Do fold 0 first; use more folds only if you have GPU time.

In [ ]:
for v in ("paper", "improved"):
    for k in range(1):          # change to range(5) for all folds
        ck = pretrain("citrus", fold=k, seed=0, variant=v)
        run_experiment("citrus", fold=k, seed=0, init_path=ck, label=v)

## 9. Summary

In [ ]:
BASELINE = {"plantdoc": 28.97, "citrus": 90.61}     # your measured from-scratch baseline (50 epochs)
df = pd.read_csv(RESULTS)
print(df[["tag", "best_epoch", "val_acc", "test_acc", "test_macro_f1", "test_minority_f1", "minutes"]].round(2).to_string(index=False))
print()
g = df.groupby(["dataset", "label"])[["test_acc", "test_macro_f1", "test_minority_f1"]].agg(["mean", "std"]).round(2)
print(g.to_string())
print()
for ds, b in BASELINE.items():
    sub = df[df.dataset == ds].groupby("label").test_acc.mean()
    for lab, v in sub.items():
        print(f"{ds:9s} {lab:18s} {v:6.2f}  vs baseline {b:6.2f}  ->  {v-b:+.2f} points")

## What to send me
- After the quick test: the SSL epoch lines (with time and entropy), the kNN lines, and the `RESULT` line, or the full error
- After real runs: the summary table